# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata.to_json()

print(f"{metadata['name']}: {metadata['description']}")
print(f"Dataset version: {metadata.get('version')}")
print(f"Published: {metadata.get('datePublished')}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

Next, we inspect the record sets available in the dataset. Each entity is referenced by its `@id` as per the Croissant schema standard.

In [ ]:
# Print all record sets and their field @ids
record_sets_list = dataset.metadata.record_sets
print(f"Available record sets ({len(record_sets_list)}):")
for rs in record_sets_list:
    print(f"- RecordSet @id: {rs['@id']} | name: {rs.get('name', '')}")
    print("  Fields:")
    for field in rs.get('fields', []):
        print(f"    - Field @id: {field['@id']} | name: {field.get('name', '')} | dataType: {field.get('dataType', '')}")
    print("")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

Let's extract all records from the key survey record set in this dataset, referenced by its `@id`.

In [ ]:
# Gather list of record set @ids for extraction
record_set_ids = [rs['@id'] for rs in dataset.metadata.record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Extracted {len(df)} rows for RecordSet @id: {record_set_id}")

# Select the principal survey record set and display its columns
# If only one, use it; otherwise, select the main one by name or index
main_record_set_id = record_set_ids[0]
print("Available columns (fields) for the main record set:", dataframes[main_record_set_id].columns.tolist())
dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes operations like removing outliers, transforming data distributions, and grouping data by key attributes to prepare it for further analysis.

We refer to fields using their `@id` from the schema. Let's demonstrate EDA by filtering for respondents with PHQ-9 scores above a certain threshold, normalizing scores, and grouping by demographic field (e.g., gender).

In [ ]:
# Use explicit field @ids from record set
main_rs = dataset.metadata.record_sets[0]
# Find numeric field: PHQ-9 score (assume field name or id contains phq9_score)
phq9_field_id = None
gender_field_id = None

for field in main_rs['fields']:
    fname = field.get('name', '').lower()
    if 'phq-9' in fname or 'phq9' in fname or 'phq9_score' in fname:
        phq9_field_id = field['@id']
    if 'gender' in fname:
        gender_field_id = field['@id']

if phq9_field_id is not None:
    print(f"PHQ-9 score field @id: {phq9_field_id}")
else:
    phq9_field_id = dataframes[main_record_set_id].columns[0]  # fallback
    print("Fallback PHQ-9 field @id:", phq9_field_id)
if gender_field_id is not None:
    print(f"Gender field @id: {gender_field_id}")
else:
    gender_field_id = dataframes[main_record_set_id].columns[1]  # fallback
    print("Fallback gender field @id:", gender_field_id)

# Filter for PHQ-9 score > 10
threshold = 10
numeric_field = phq9_field_id

filtered_df = dataframes[main_record_set_id][dataframes[main_record_set_id][numeric_field] > threshold]
print(f"Filtered records with {numeric_field} > {threshold}:")
print(filtered_df.head())

# Normalize the PHQ-9 scores
filtered_df[f"{numeric_field}_normalized"] = (
    filtered_df[numeric_field] - filtered_df[numeric_field].mean()
) / filtered_df[numeric_field].std()
print(f"Normalized {numeric_field} for filtered records:")
print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Group by gender and show means
group_field = gender_field_id
if group_field in dataframes[main_record_set_id].columns:
    grouped_df = filtered_df.groupby(group_field, as_index=False).agg({numeric_field: 'mean', f"{numeric_field}_normalized": 'mean'})
    print(f"Grouped data by {group_field}:")
    print(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

Below, we plot the distribution of PHQ-9 scores among respondents and visualize the average scores by gender.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Distribution plot of PHQ-9 scores
sns.histplot(dataframes[main_record_set_id][phq9_field_id].dropna(), bins=15, kde=True)
plt.xlabel("PHQ-9 Score")
plt.title("Distribution of PHQ-9 Scores")
plt.show()

# Bar plot: Mean PHQ-9 scores by gender
if gender_field_id in dataframes[main_record_set_id].columns:
    gender_means = dataframes[main_record_set_id].groupby(gender_field_id)[phq9_field_id].mean().reset_index()
    sns.barplot(x=gender_field_id, y=phq9_field_id, data=gender_means)
    plt.title("Mean PHQ-9 Score by Gender")
    plt.ylabel("Mean PHQ-9 Score")
    plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- Loaded and explored the FAIR² Mental Health Survey Dataset using `mlcroissant`.
- Identified and referenced record sets and fields via their `@id`.
- Filtered survey respondents based on PHQ-9 scores and normalized values for further analysis.
- Grouped results by demographic attributes for deeper insights.
- Visualized score distributions and demographic patterns.

For further exploration, additional fields and metrics (GAD-7, PSQ) may be analyzed similarly using their respective `@id` references in the schema.